In [13]:
import pandas as pd
import numpy as np

In [14]:
# Load RFM analysis results
rfm = pd.read_csv("../outputs/tables/customer_rfm.csv")

# Load K-Means segmentation results
segments = pd.read_csv("../outputs/tables/customer_segments.csv")


In [15]:
print("RFM shape:", rfm.shape)
print("Segments shape:", segments.shape)

print("\nRFM columns:")
print(rfm.columns.tolist())

print("\nSegment columns:")
print(segments.columns.tolist())

RFM shape: (7, 5)
Segments shape: (77546, 21)

RFM columns:
['rfm_segment', 'customers', 'avg_recency', 'avg_frequency', 'avg_monetary']

Segment columns:
['user_id', 'total_orders', 'total_spend', 'total_quantity', 'average_order_value', 'first_order_date', 'last_order_date', 'age', 'gender', 'marital_status', 'occupation', 'recency', 'frequency', 'monetary', 'R_score', 'F_score', 'M_score', 'RFM_score', 'RFM_total', 'rfm_segment', 'cluster']


CHECKING THE CUSTOMERS SEGMENTS

In [16]:
print(segments[["user_id","rfm_segment","cluster"]].head(10))

print("\nRFM Segment counts:")
print(segments["rfm_segment"].value_counts())

print("\nK-Means Cluster counts:")
print(segments["cluster"].value_counts().sort_index())

   user_id           rfm_segment  cluster
0        1   Potential Loyalists        0
1        2  At Risk - High Value        0
2        3   Potential Loyalists        0
3        4      Lost / Low Value        1
4        5  At Risk - High Value        0
5        6      Lost / Low Value        1
6        8   Potential Loyalists        3
7        9   Potential Loyalists        2
8       10       New / Promising        3
9       13   Potential Loyalists        2

RFM Segment counts:
rfm_segment
Potential Loyalists         27252
New / Promising             19513
Lost / Low Value            15211
At Risk - High Value         9285
Champions                    3530
Loyal Customers              2139
At Risk - High Frequency      616
Name: count, dtype: int64

K-Means Cluster counts:
cluster
0    15135
1    17927
2    23595
3    20889
Name: count, dtype: int64


In [17]:
import joblib

# Load the trained Random Forest churn model
churn_model = joblib.load("../models/churn_random_forest.pkl")

print("Churn model loaded successfully!")

Churn model loaded successfully!


PREPARE CUSTOMER FEATURES FOR CHURN PREDICTIONS

In [18]:
churn_features = segments[["user_id","total_orders","total_spend","total_quantity","average_order_value","recency"]].copy()

# Create the remaining features required by the churn model
segments["first_order_date"] = pd.to_datetime(segments["first_order_date"])
segments["last_order_date"] = pd.to_datetime(segments["last_order_date"])

segments["tenure_days"] = (segments["last_order_date"] -segments["first_order_date"]).dt.days

segments["active_months"] = (segments["tenure_days"] / 30).round(2)

segments["average_days_between_orders"] = np.where(segments["total_orders"] > 1,
                segments["tenure_days"] / (segments["total_orders"] - 1),0)

segments["orders_per_month"] = np.where(segments["active_months"] > 0,segments["total_orders"] / segments["active_months"],
    segments["total_orders"])

features_for_prediction = segments[["total_orders","total_spend","total_quantity","average_order_value","recency",
        "tenure_days","active_months","average_days_between_orders","orders_per_month"]]

print("Feature shape:", features_for_prediction.shape)
print("\nFeatures used:")
print(features_for_prediction.columns.tolist())

Feature shape: (77546, 9)

Features used:
['total_orders', 'total_spend', 'total_quantity', 'average_order_value', 'recency', 'tenure_days', 'active_months', 'average_days_between_orders', 'orders_per_month']


GERATING CHURN PROBABILITY

In [19]:
# Predict churn probability
churn_probability = churn_model.predict_proba(features_for_prediction)[:, 1]

# Add probability to the customer data
segments["churn_probability"] = churn_probability

print(segments[["user_id", "churn_probability"]].head(10))

   user_id  churn_probability
0        1           0.810000
1        2           0.820000
2        3           0.830000
3        4           0.401057
4        5           0.920000
5        6           0.990000
6        8           0.970000
7        9           0.850000
8       10           0.800000
9       13           0.640000


CREATING CHURN RISK LEVELS

In [49]:
# Rule-based customer risk assessment
# Based on recency and order frequency, not ML predictions

def assign_risk(row):
    recency = row["recency"]
    frequency = row["total_orders"]

    if frequency == 0:
        return "Not Yet Rated"
    elif recency <= 30 and frequency >= 3:
        return "Low Risk"
    elif recency <= 60:
        return "Medium Risk"
    else:
        return "High Risk"


segments["churn_risk"] = segments.apply(assign_risk, axis=1)

print("Rule-based Churn Risk Distribution:")
print(segments["churn_risk"].value_counts())

risk_by_segment = pd.crosstab(
    segments["rfm_segment"],
    segments["churn_risk"]
)

print(risk_by_segment)

Rule-based Churn Risk Distribution:
churn_risk
High Risk      70679
Medium Risk     5459
Low Risk        1408
Name: count, dtype: int64
churn_risk                High Risk  Low Risk  Medium Risk
rfm_segment                                               
At Risk - High Frequency        616         0            0
At Risk - High Value           9285         0            0
Champions                      2558       474          498
Lost / Low Value              15211         0            0
Loyal Customers                1886       117          136
New / Promising               15548         0         3965
Potential Loyalists           25575       817          860


COMBINE RFM SEGMENT AND CHURN RISK

In [21]:
risk_by_segment = pd.crosstab(segments["rfm_segment"],segments["churn_risk"])

print(risk_by_segment)

churn_risk                High Risk  Low Risk  Medium Risk
rfm_segment                                               
At Risk - High Frequency        607         0            9
At Risk - High Value           8395        27          863
Champions                      3178         8          344
Lost / Low Value              11652       767         2792
Loyal Customers                2020         0          119
New / Promising               16633       467         2413
Potential Loyalists           24369       371         2512


CREATING CUSTOMER RECOMMENDATIONS

In [22]:
def get_recommendation(row):

    segment = row["rfm_segment"]
    risk = row["churn_risk"]

    if segment == "Champions" and risk == "High Risk":
        return "VIP retention offer"

    elif segment == "Loyal Customers" and risk == "High Risk":
        return "Personalized retention offer"

    elif segment == "At Risk - High Value":
        return "High-value win-back campaign"

    elif segment == "At Risk - High Frequency":
        return "Retention coupon"

    elif segment == "Potential Loyalists" and risk == "High Risk":
        return "Encourage repeat purchase"

    elif segment == "New / Promising":
        return "New-customer engagement offer"

    elif segment == "Lost / Low Value":
        return "Win-back campaign"

    elif risk == "Medium Risk":
        return "Engagement campaign"

    else:
        return "Maintain customer relationship"


segments["recommendation"] = segments.apply(get_recommendation,axis=1)

print(segments[["user_id","rfm_segment","churn_risk","recommendation"]].head(15))

    user_id           rfm_segment   churn_risk                 recommendation
0         1   Potential Loyalists    High Risk      Encourage repeat purchase
1         2  At Risk - High Value    High Risk   High-value win-back campaign
2         3   Potential Loyalists    High Risk      Encourage repeat purchase
3         4      Lost / Low Value  Medium Risk              Win-back campaign
4         5  At Risk - High Value    High Risk   High-value win-back campaign
5         6      Lost / Low Value    High Risk              Win-back campaign
6         8   Potential Loyalists    High Risk      Encourage repeat purchase
7         9   Potential Loyalists    High Risk      Encourage repeat purchase
8        10       New / Promising    High Risk  New-customer engagement offer
9        13   Potential Loyalists  Medium Risk            Engagement campaign
10       14  At Risk - High Value    High Risk   High-value win-back campaign
11       15   Potential Loyalists    High Risk      Encourage re

CREATING THE FINAL CUSTOMER RISK TABLE

In [51]:
customer_risk = segments[
    [
        "user_id",
        "total_orders",
        "total_spend",
        "average_order_value",
        "rfm_segment",
        "cluster",
        "churn_risk",
        "recommendation"
    ]
].copy()

print("Final table shape:", customer_risk.shape)
print(customer_risk.head())

Final table shape: (77546, 8)
   user_id  total_orders  total_spend  average_order_value  \
0        1             2         1672           836.000000   
1        2             3         4061          1353.666667   
2        3             1        21542         21542.000000   
3        4             1          102           102.000000   
4        5             2        52047         26023.500000   

            rfm_segment  cluster churn_risk                recommendation  
0   Potential Loyalists        0  High Risk     Encourage repeat purchase  
1  At Risk - High Value        0  High Risk  High-value win-back campaign  
2   Potential Loyalists        0  High Risk     Encourage repeat purchase  
3      Lost / Low Value        1  High Risk             Win-back campaign  
4  At Risk - High Value        0  High Risk  High-value win-back campaign  


In [56]:
customer_risk.to_csv(
    "../outputs/tables/customer_risk_recommendations.csv",
    index=False
)

print("Final customer risk dataset saved successfully!")
print("Rows:", customer_risk.shape[0])
print("Columns:", customer_risk.shape[1])

Final customer risk dataset saved successfully!
Rows: 77546
Columns: 8


In [57]:
recommendation_counts = (customer_risk["recommendation"].value_counts().reset_index())

recommendation_counts.columns = ["recommendation","customers"]
print(recommendation_counts)

                   recommendation  customers
0       Encourage repeat purchase      24369
1   New-customer engagement offer      19513
2               Win-back campaign      15211
3    High-value win-back campaign       9285
4             VIP retention offer       3178
5             Engagement campaign       2975
6    Personalized retention offer       2020
7                Retention coupon        616
8  Maintain customer relationship        379


CREATING A BUSINESS SUMMARY

In [58]:
business_summary = (customer_risk.groupby(["rfm_segment", "churn_risk"]).agg(
        customers=("user_id", "count"),total_spend=("total_spend", "sum"),
        avg_order_value=("average_order_value", "mean")).reset_index())

print(business_summary)

                 rfm_segment   churn_risk  customers  total_spend  \
0   At Risk - High Frequency    High Risk        616     14981627   
1       At Risk - High Value    High Risk       9285    265665070   
2                  Champions    High Risk       2558     96175407   
3                  Champions     Low Risk        474     18761706   
4                  Champions  Medium Risk        498     16784337   
5           Lost / Low Value    High Risk      15211      5213982   
6            Loyal Customers    High Risk       1886     34850698   
7            Loyal Customers     Low Risk        117       230378   
8            Loyal Customers  Medium Risk        136       266517   
9            New / Promising    High Risk      15548    157739678   
10           New / Promising  Medium Risk       3965     41280695   
11       Potential Loyalists    High Risk      25575    280297152   
12       Potential Loyalists     Low Risk        817     15563248   
13       Potential Loyalists  Medi

In [59]:
print("Missing values:")
print(customer_risk.isnull().sum())

print("\nDuplicate user IDs:")
print(customer_risk["user_id"].duplicated().sum())

print("\nTotal customers:")
print(customer_risk["user_id"].nunique())

print("\nFinal shape:")
print(customer_risk.shape)

Missing values:
user_id                0
total_orders           0
total_spend            0
average_order_value    0
rfm_segment            0
cluster                0
churn_risk             0
recommendation         0
dtype: int64

Duplicate user IDs:
0

Total customers:
77546

Final shape:
(77546, 8)


In [60]:
customer_risk.to_csv("../outputs/tables/customer_risk_recommendations.csv",index=False)

print("Final customer risk dataset saved successfully!")
print("File: customer_risk_recommendations.csv")
print("Rows:", customer_risk.shape[0])
print("Columns:", customer_risk.shape[1])

Final customer risk dataset saved successfully!
File: customer_risk_recommendations.csv
Rows: 77546
Columns: 8


In [61]:
# Create the RFM population DataFrame
rfm_population = rfm.copy()

print("RFM population shape:", rfm_population.shape)
print("Columns:", rfm_population.columns.tolist())

RFM population shape: (7, 5)
Columns: ['rfm_segment', 'customers', 'avg_recency', 'avg_frequency', 'avg_monetary']


In [62]:
rfm_population = pd.read_csv(
    "../outputs/tables/customer_rfm_individual.csv"
)

print("Shape:", rfm_population.shape)
print("Columns:", rfm_population.columns.tolist())

Shape: (77546, 9)
Columns: ['recency', 'frequency', 'monetary', 'F_score', 'M_score', 'R_score', 'RFM_score', 'RFM_total', 'rfm_segment']


In [63]:
rfm_population["F_score"] = pd.qcut(
    rfm_population["frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm_population["M_score"] = pd.qcut(
    rfm_population["monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)
rfm_population["R_score"] = pd.qcut(
    rfm_population["recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
)

print(rfm_population[["frequency", "monetary", "R_score","F_score", "M_score",]].head())

   frequency  monetary R_score F_score M_score
0          2      1672       2       3       3
1          3      4061       1       4       4
2          1     21542       3       1       5
3          1       102       2       1       1
4          2     52047       1       3       5


In [64]:
rfm_population["RFM_score"] = (
    rfm_population["F_score"].astype(int).astype(str)
    + rfm_population["M_score"].astype(int).astype(str)
)

print(rfm_population[["frequency", "monetary", "F_score", "M_score", "RFM_score"]].head())

   frequency  monetary F_score M_score RFM_score
0          2      1672       3       3        33
1          3      4061       4       4        44
2          1     21542       1       5        15
3          1       102       1       1        11
4          2     52047       3       5        35


In [65]:
rfm_population["R_score"] = pd.qcut(
    rfm_population["recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
)

print(
    rfm_population[
        ["recency", "frequency", "monetary", "R_score", "F_score", "M_score"]
    ].head()
)

   recency  frequency  monetary R_score F_score M_score
0      590          2      1672       2       3       3
1      693          3      4061       1       4       4
2      408          1     21542       3       1       5
3      506          1       102       2       1       1
4      702          2     52047       1       3       5


In [66]:
rfm_population["RFM_total"] = (
    rfm_population["R_score"].astype(int)
    + rfm_population["F_score"].astype(int)
    + rfm_population["M_score"].astype(int)
)

print(
    rfm_population[
        ["R_score", "F_score", "M_score", "RFM_total"]
    ].head()
)

  R_score F_score M_score  RFM_total
0       2       3       3          8
1       1       4       4          9
2       3       1       5          9
3       2       1       1          4
4       1       3       5          9


In [67]:
rfm_population["rfm_segment"] = np.select(
    [
        (rfm_population["RFM_total"] >= 13),
        (rfm_population["RFM_total"] >= 10),
        (rfm_population["RFM_total"] >= 7)
    ],
    [
        "Champions",
        "Loyal Customers",
        "Potential Loyalists"
    ],
    default="At Risk / Needs Attention"
)

print(rfm_population["rfm_segment"].value_counts())

rfm_segment
Loyal Customers              22967
Potential Loyalists          22928
At Risk / Needs Attention    19319
Champions                    12332
Name: count, dtype: int64


In [68]:
print(rfm_population.head(10))

print("\nMissing values:")
print(rfm_population.isnull().sum())

print("\nTotal customers:", len(rfm_population))

   recency  frequency  monetary F_score M_score R_score RFM_score  RFM_total  \
0      590          2      1672       3       3       2        33          8   
1      693          3      4061       4       4       1        44          9   
2      408          1     21542       1       5       3        15          9   
3      506          1       102       1       1       2        11          4   
4      702          2     52047       3       5       1        35          9   
5      623          1        83       1       1       2        11          4   
6      371          1       241       1       1       3        11          5   
7      232          3      7940       4       4       4        44         12   
8       67          2      2773       3       3       5        33         11   
9      147          3      3819       4       4       4        44         12   

                 rfm_segment  
0        Potential Loyalists  
1        Potential Loyalists  
2        Potential Loyalis

In [69]:
import os

os.makedirs("../outputs/tables", exist_ok=True)

rfm_population.to_csv(
    "../outputs/tables/customer_rfm_individual.csv",
    index=False
)

print("Individual customer RFM results saved successfully.")

Individual customer RFM results saved successfully.


In [70]:
segments.to_csv(
    "../outputs/tables/customer_risk_recommendations.csv",
    index=False
)
print("Customer risk recommendations saved successfully.")
print("Total customers:", len(segments))

Customer risk recommendations saved successfully.
Total customers: 77546


In [71]:
customer_risk.to_csv(
    "../outputs/tables/customer_risk_recommendations.csv",
    index=False
)

print("Final customer risk dataset saved successfully!")
print("Rows:", customer_risk.shape[0])
print("Columns:", customer_risk.shape[1])

Final customer risk dataset saved successfully!
Rows: 77546
Columns: 8
